# Tutorial: Complex Geometries

This tutorial covers multi-compartment geometries, importing geometries from existing models, and analytic geometry primitives.

See also the [Complex Geometries](../complex-geometries.md) reference guide.

## Define a multi-compartment model

In [1]:
import pyvcell.vcml as vc

antimony_str = """
    compartment ec = 10000;
    compartment cell = 5000;
    compartment pm = 100;
    compartment nuc = 300;
    compartment nuc_env = 40;
    species A in cell;
    species B in cell;
    J0: A -> B; cell * (k1*A - k2*B)
    J0 in cell;
    k1 = 5.0; k2 = 2.0
    A = 10
"""

biomodel = vc.load_antimony_str(antimony_str)
model = biomodel.model
model.get_compartment("pm").dim = 2
model.get_compartment("nuc_env").dim = 2
print(model)

2026-03-08T03:59:10.139202Z main WARN The use of package scanning to locate Log4j plugins is deprecated.
Please remove the `packages` attribute from your configuration file.
See https://logging.apache.org/log4j/2.x/faq.html#package-scanning for details.
2026-03-07 22:59:10,142 ERROR (SBMLDocument.java:573) - There was an error accessing the sbml online validator!
2026-03-08T03:59:10.146960Z main WARN The Logger cbit.vcell.model.Kinetics was created with the message factory org.apache.logging.log4j.message.ReusableMessageFactory@32499e7a and is now requested with a null message factory (defaults to org.apache.logging.log4j.message.ParameterizedMessageFactory), which may create log events with unexpected formatting.
2026-03-08T03:59:10.149649Z main WARN The Logger cbit.vcell.mapping.AbstractMathMapping was created with the message factory org.apache.logging.log4j.message.ReusableMessageFactory@32499e7a and is now requested with a null message factory (defaults to org.apache.logging.log4j

## Load geometry from an existing model

In [2]:
tutorial_biomodel = vc.load_vcml_url(
    "https://raw.githubusercontent.com/virtualcell/pyvcell/refs/heads/main/"
    "examples/models/Tutorial_MultiApp_PDE.vcml"
)

tutorial_geometry = tutorial_biomodel.applications[0].geometry

print("Subvolumes:", tutorial_geometry.subvolume_names)
print("Surfaces:", tutorial_geometry.surface_class_names)

Subvolumes: ['ec', 'cytosol', 'Nucleus']
Surfaces: ['cytosol_ec_membrane', 'Nucleus_cytosol_membrane']


## Visualize the imported geometry

In [3]:
tutorial_geometry.plot(save_path="../images/complex-geometry.png")

/Users/jimschaff/Documents/workspace/pyvcell/pyvcell/_internal/geometry/segmented_image_geometry.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Map compartments and simulate

In [4]:
app = biomodel.add_application("app1", geometry=tutorial_geometry)

app.map_compartment("cell", "cytosol")
app.map_compartment("ec", "ec")
app.map_compartment("nuc", "Nucleus")
app.map_compartment("nuc_env", "Nucleus_cytosol_membrane")
app.map_compartment("pm", "cytosol_ec_membrane")

app.map_species("A", init_conc="3+sin(0.2*x)", diff_coef=1.0)
app.map_species("B", init_conc="2+cos(0.2*(x+y+z))", diff_coef=1.0)

sim = app.add_sim(name="sim1", duration=2.0, output_time_step=0.05, mesh_size=(50, 50, 50))
results = vc.simulate(biomodel=biomodel, simulation="sim1")

2026-03-08T03:59:11.262197Z main WARN The use of package scanning to locate Log4j plugins is deprecated.
Please remove the `packages` attribute from your configuration file.
See https://logging.apache.org/log4j/2.x/faq.html#package-scanning for details.
2026-03-08T03:59:11.265816Z main WARN The Logger cbit.vcell.model.Kinetics was created with the message factory org.apache.logging.log4j.message.ReusableMessageFactory@4d4cbc68 and is now requested with a null message factory (defaults to org.apache.logging.log4j.message.ParameterizedMessageFactory), which may create log events with unexpected formatting.


2026-03-08T03:59:11.572163Z main WARN The Logger cbit.vcell.mapping.AbstractMathMapping was created with the message factory org.apache.logging.log4j.message.ReusableMessageFactory@4d4cbc68 and is now requested with a null message factory (defaults to org.apache.logging.log4j.message.ParameterizedMessageFactory), which may create log events with unexpected formatting.
2026-03-07 22:59:11,573  INFO (DiffEquMathMapping.java:1457) - WARNING:::: MathMapping.refreshMathDescription() ... assigning boundary condition types not unique
2026-03-07 22:59:11,573  INFO (DiffEquMathMapping.java:1457) - WARNING:::: MathMapping.refreshMathDescription() ... assigning boundary condition types not unique
2026-03-07 22:59:11,579  INFO (Entrypoints.java:200) - Returning from vcellToVcml: {"success":true,"message":"Success"}
2026-03-08T03:59:11.595128Z main WARN The use of package scanning to locate Log4j plugins is deprecated.
Please remove the `packages` attribute from your configuration file.
See https:/

Simulation Complete in Main() ... 


initializing mesh
numVolume=125000

CartesianMesh::computeNormalsFromNeighbors(), compute normals from neighbors
Membrane Elements -> N=8306
qhull precision warning: 
859 has 0 neighbors !
6830 has 0 neighbors !
--------Num of points that have zero neighbors 2
--------Num Neighbors before symmetrize 48504
--------Num Neighbors after symmetrize 53758
Total volume=5921.705897
Total FluxArea =50.44805177
Total FluxAreaXM =0
Total FluxAreaXP =0
Total FluxAreaYM =34.01199592
Total FluxAreaYP =16.43605585
Total FluxAreaZM =0
Total FluxAreaZP =0
mesh initialized
preprocessing finished
pdeCount=2, odeCount=0
No log-file found at constructed path `/Users/jimschaff/Documents/workspace/pyvcell/docs/guides/notebooks/workspace/out_dir_16af6tl4/SimID_600541721_0_.log`.simulation [SimID_600541721_0_] started
temporary directory used is /var/folders/zz/gcfcvgtd5v1cdgj2sw4bjzdr0000gr/T/
sim file name is /var/folders/zz/gcfcvgtd5v1cdgj2sw4bjzdr0000gr/T/SimID_600541721_0_0000.sim
**This is a little endia

In [5]:
results.plotter.plot_concentrations(save_path="../images/complex-concentrations.png")

/Users/jimschaff/Documents/workspace/pyvcell/pyvcell/sim_results/plotter.py:71: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  return plt.show()


In [6]:
results.plotter.plot_slice_3d(time_index=0, channel_id="A", save_path="../images/complex-slice3d-A.png")

/Users/jimschaff/Documents/workspace/pyvcell/pyvcell/sim_results/plotter.py:138: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  return plt.show()


## Analytic geometry primitives

You can also build geometries from scratch using analytic helpers.

In [7]:
geo = vc.Geometry(name="geo", origin=(0, 0, 0), extent=(10, 10, 10), dim=3)
geo.add_sphere(name="cell_domain", radius=4, center=(5, 5, 5))
geo.add_background(name="ec_domain")
geo.add_surface(name="pm_domain", sub_volume_1="cell_domain", sub_volume_2="ec_domain")

geo.plot(save_path="../images/complex-analytic-geometry.png")

## Clean up

In [8]:
results.cleanup()